In [2]:
"""
Multi-Agent Financial Analysis System
Team 7: Andrea Thomas, Jinyuan He, Thiago Alvaraes
GitHub Repository: https://github.com/Yuan7Code/AAI520-Group-7

# ==========================================
# AI USAGE DISCLOSURE
# ==========================================
In accordance with the USD Code of Honor and course policies, we disclose the following use of AI tools (Gemini / ChatGPT / GitHub Copilot) in this project:
- Code Generation: AI was used to assist in scaffolding the LangGraph state machine and drafting prompt templates for the LLM agents. All AI-generated code was manually reviewed, modified to fit external APIs, and tested for accuracy.
- Troubleshooting: AI was used to resolve environment issues with package dependencies and API rate limits.

# ==========================================
# AGENT DESIGN, WORKFLOWS & CAPABILITIES
# ==========================================
- Agent Functions: The Investment Research Agent uses two dynamic tools (`fetch_stock_info` and `fetch_recent_news`) to gather live financial data and news sentiment[cite: 2, 3, 8]. It reflects on its own outputs via the `critique` node, and saves persistent memory of its runs via the `save_memory` function[cite: 2, 8].
- Prompt Chaining Pattern: Raw text is ingested via NewsAPI, preprocessed to remove HTML, classified by relevance, extracted for key facts, and summarized into domain-specific contexts[cite: 2, 8].
- Routing & Evaluator-Optimizer: A router dynamically directs the summarized text to specialized analysts (Earnings, News, Market). An independent Evaluator grades the analysis against the raw evidence and triggers a revision loop if the score falls below 0.75[cite: 2, 8].
"""

import os
import re
import json
import time
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import TypedDict, List, Literal

import requests
import yfinance as yf
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END

# --- API KEY SETUP FOR COLAB ---
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
os.environ["NEWSAPI_KEY"] = userdata.get("NEWSAPI_KEY")

client = genai.Client()
MODEL = "gemini-3.5-flash"

# ==========================================
# 1. DATA AND TOOL FUNCTIONS (Andrea)
# ==========================================
def fetch_stock_info(symbol: str) -> dict:
    """Retrieves core financial metrics and company information via yfinance."""
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.info
        return {
            "symbol": symbol,
            "company_name": info.get("shortName", "N/A"),
            "current_price": info.get("currentPrice", "N/A"),
            "market_cap": info.get("marketCap", "N/A")
        }
    except Exception as e:
        return {"error": str(e)}

def fetch_recent_news(symbol: str) -> list[dict]:
    """Retrieves live market news using NewsAPI for the prompt chaining workflow."""
    api_key = os.environ.get("NEWSAPI_KEY")
    if not api_key:
        print(f"[NewsAPI] Error: NEWSAPI_KEY is not set in environment.")
        return []

    url = f"https://newsapi.org/v2/everything?q={symbol}+stock&sortBy=publishedAt&language=en&pageSize=10&apiKey={api_key}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()

        if data.get("status") != "ok":
            print(f"[NewsAPI] API error: {data.get('message', 'Unknown error')}")
            return []

        formatted_news = []
        for item in data.get("articles", [])[:10]:
            formatted_news.append({
                "title": item.get("title") or "",
                "source": (item.get("source") or {}).get("name") or "Unknown",
                "published_at": item.get("publishedAt") or datetime.now(timezone.utc).isoformat(),
                "url": item.get("url") or "",
                "text": item.get("description") or item.get("content") or ""
            })
        return formatted_news
    except Exception as e:
        print(f"[NewsAPI] Request failed for {symbol}: {e}")
        return []


# ==========================================
# 2. PROMPT CHAINING & AGENTS (Jinyuan & Thiago)
# ==========================================
class State(TypedDict):
    ticker: str
    plan: List[str]
    raw_news: list
    stock_info: dict  # Added to utilize the yfinance tool data dynamically
    routed: dict
    analysis: str
    critique: str
    score: float
    iterations: int

class Plan(BaseModel):
    steps: List[str]

CATEGORIES = ("earnings", "news", "market")
CHAIN_SYSTEM = "You are a careful equity research analyst. Use only the information provided."

class Classified(BaseModel):
    id: int
    category: Literal["earnings", "news", "market"]
    reason: str

class Classification(BaseModel):
    items: List[Classified]

class Extracted(BaseModel):
    id: int
    sentiment: Literal["positive", "negative", "neutral"]
    key_facts: List[str]
    impact: str

class Extraction(BaseModel):
    items: List[Extracted]

class Summary(BaseModel):
    earnings: str
    news: str
    market: str

def ask(prompt: str, schema=None, system: str = CHAIN_SYSTEM):
    """Executes a Gemini call with exponential backoff to handle transient 503 errors."""
    if schema is None:
        config = types.GenerateContentConfig(system_instruction=system)
    else:
        config = types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=schema,
        )

    for attempt in range(5):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
            if schema is None:
                if not response.text:
                    raise RuntimeError("Text step returned an empty response")
                return response.text.strip()
            if response.parsed is None:
                raise RuntimeError(f"{schema.__name__} step returned invalid output: {response.text}")
            return response.parsed
        except Exception as e:
            if "503" in str(e) and attempt < 4:
                sleep_time = (2 ** attempt) + random.uniform(0.5, 1.5)
                print(f"[API] 503 high demand. Retrying in {sleep_time:.1f}s...")
                time.sleep(sleep_time)
            else:
                raise e

def clean_text(text: str, max_chars: int = 1500) -> str:
    text = re.sub(r"<[^>]+>", " ", text or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]

def planner(state: State) -> dict:
    """Agent Function: Plans research steps[cite: 2]."""
    ticker = state["ticker"].upper()
    prompt = (
        f"Create a 4-6 step research plan to analyze the stock {ticker}. "
        "Cover data gathering (news, earnings, price), analysis, and self-critique. "
        "Keep each step short and actionable."
    )
    plan_obj = ask(
        prompt,
        schema=Plan,
        system="You are an investment research agent that plans before acting."
    )
    return {"ticker": ticker, "plan": plan_obj.steps, "iterations": 0}

def fetch(state: State) -> dict:
    """Agent Function: Uses tools dynamically (APIs, datasets)[cite: 2, 3]."""
    ticker = state["ticker"]

    # Tool 1: yfinance for quantitative metrics
    stock_info = fetch_stock_info(ticker)

    # Tool 2: NewsAPI for qualitative sentiment
    items = fetch_recent_news(ticker)

    seen, raw_news = set(), []
    for item in sorted(items, key=lambda x: x["published_at"], reverse=True):
        if item["title"] not in seen:
            seen.add(item["title"])
            raw_news.append(item)

    print(f"[fetch] Retrieved metrics & {len(raw_news)} live articles for {ticker}")
    return {"raw_news": raw_news, "stock_info": stock_info}

def chain(state: State) -> dict:
    """Workflow Pattern: Prompt Chaining (Ingest -> Preprocess -> Classify -> Extract -> Summarize)[cite: 2]."""
    ticker = state["ticker"]
    routed = {c: {"items": [], "summary": ""} for c in CATEGORIES}

    # 1. Preprocess
    articles = []
    for n in state["raw_news"]:
        title, text = clean_text(n.get("title"), 300), clean_text(n.get("text"))
        if title or text:
            articles.append({
                "id": len(articles),
                "title": title,
                "text": text,
                "source": n.get("source", ""),
                "published_at": n.get("published_at", ""),
                "url": n.get("url", "")
            })
    if not articles:
        print("[chain] No articles available to process.")
        return {"routed": routed}

    listing = "\n\n".join(f"[{a['id']}] {a['title']}\n{a['text']}" for a in articles)

    # 2. Classify
    classification = ask(
        f"Classify each article about {ticker} into exactly one category:\n"
        "- earnings: the company's financial results, guidance, EPS, revenue\n"
        "- news: company-specific events (products, deals, legal/regulatory, analyst ratings, supply chain)\n"
        "- market: macro or sector-wide moves (interest rates, the Fed, indices, sector rotation)\n\n"
        f"Articles:\n{listing}",
        Classification
    )
    category_of = {c.id: c.category for c in classification.items}

    # 3. Extract
    labeled = "\n\n".join(
        f"[{a['id']}] ({category_of.get(a['id'], 'news')}) {a['title']}\n{a['text']}" for a in articles
    )
    extraction = ask(
        f"For each article, extract: the sentiment for {ticker} stock, 1-3 key facts "
        f"(keep any numbers), and a one-sentence likely impact on {ticker}'s stock price.\n\n"
        f"Articles:\n{labeled}",
        Extraction
    )
    details = {e.id: e for e in extraction.items}

    for a in articles:
        cat = category_of.get(a["id"], "news")
        e = details.get(a["id"])
        a.update(
            category=cat,
            sentiment=e.sentiment if e else "neutral",
            key_facts=e.key_facts if e else [],
            impact=e.impact if e else ""
        )
        routed[cat]["items"].append(a)

    # 4. Summarize
    facts = "\n\n".join(
        f"{cat.upper()}:\n" + ("\n".join(
            f"- [{a['sentiment']}] {'; '.join(a['key_facts'])} -> {a['impact']}"
            for a in routed[cat]["items"]) or "- (no articles)")
        for cat in CATEGORIES
    )
    summary = ask(
        f"Using only these extracted facts about {ticker}, write a 2-3 sentence summary "
        "for each category. For a category with no articles, write 'No relevant articles.'\n\n"
        f"{facts}",
        Summary
    )
    for cat in CATEGORIES:
        routed[cat]["summary"] = getattr(summary, cat)
    return {"routed": routed}

def pick_route(routed: dict) -> str:
    """Workflow Pattern: Routing (Direct content to the right specialist)[cite: 2]."""
    if routed["earnings"]["items"]:
        return "earnings"
    def strength(cat):
        items = routed[cat]["items"]
        pos = sum(a["sentiment"] == "positive" for a in items)
        neg = sum(a["sentiment"] == "negative" for a in items)
        return (len(items), abs(pos - neg))
    return max(("news", "market"), key=strength)

def router(state: State) -> str:
    chosen_route = pick_route(state["routed"])
    print(f"[router] -> Routed to {chosen_route} specialist")
    return chosen_route

ANALYST_FOCUS = {
    "earnings": "You are an earnings analyst. Focus on results versus expectations, margins, and guidance.",
    "news":     "You are an equity news analyst. Focus on company-specific events, products, and legal moves.",
    "market":   "You are a macro strategist. Focus on interest rates, sector moves, and broader market sentiment.",
}

def make_analyst(category: str):
    def analyst(state: State) -> dict:
        ticker = state["ticker"]
        routed = state["routed"]
        stock_info = state.get("stock_info", {})

        overview = "\n".join(f"- {c}: {routed[c]['summary'] or 'No relevant articles.'}" for c in CATEGORIES)
        details = "\n".join(
            f"- [{a['sentiment']}] {a['title']} ({a['source']})\n  facts: {'; '.join(a['key_facts'])}\n  impact: {a['impact']}"
            for a in routed[category]["items"]
        ) or "- (no articles)"

        prompt = (
            f"Write an investment analysis of {ticker} focused on the {category} evidence.\n\n"
            f"Company Data (yfinance): Price: {stock_info.get('current_price')}, Market Cap: {stock_info.get('market_cap')}\n\n"
            f"News Overview (NewsAPI):\n{overview}\n\nDetails:\n{details}\n\n"
            "Use these sections: Thesis, Key drivers, Risks, Outlook."
        )
        if state.get("critique"):
            prompt += f"\n\nRevise your previous analysis to address this feedback:\n{state['critique']}\n\nPrevious analysis:\n{state['analysis']}"

        analysis = ask(prompt, system=ANALYST_FOCUS[category])
        return {"analysis": analysis}
    analyst.__name__ = f"{category}_analyst"
    return analyst

earnings_analyst = make_analyst("earnings")
news_analyst = make_analyst("news")
market_analyst = make_analyst("market")

PASS_SCORE = 0.75
MAX_REVISIONS = 2
CRITIC_SYSTEM = "You are a strict senior investment reviewer. Ground evaluation strictly in provided evidence."

class Critique(BaseModel):
    grounded: int = Field(ge=0, le=5)
    risks: int = Field(ge=0, le=5)
    reasoning: int = Field(ge=0, le=5)
    clarity: int = Field(ge=0, le=5)
    issues: List[str]
    feedback: str

def critique(state: State) -> dict:
    """Agent Function / Workflow Pattern: Self-reflects and Evaluator-Optimizer loop[cite: 2]."""
    ticker, routed = state["ticker"], state["routed"]
    evidence = "\n".join(
        f"- ({cat}) [{a['sentiment']}] {a['title']}: {'; '.join(a['key_facts'])}"
        for cat in CATEGORIES for a in routed[cat]["items"]
    ) or "- (no articles)"

    review = ask(
        f"Evaluate this analysis of {ticker}.\n\nEvidence:\n{evidence}\n\nAnalysis:\n{state['analysis']}\n\n",
        Critique,
        system=CRITIC_SYSTEM
    )
    parts = [review.grounded, review.risks, review.reasoning, review.clarity]
    # Fixed the missing multiplication operator in Jinyuan's original code
    score = round(sum(min(max(x, 0), 5) for x in parts) / (5 * len(parts)), 2)
    iterations = state.get("iterations", 0) + 1

    feedback = review.feedback
    if review.issues:
        feedback = "Issues:\n" + "\n".join(f"- {i}" for i in review.issues) + f"\n\n{feedback}"
    print(f"[critique] Round {iterations}: Score {score:.2f}")
    return {"critique": feedback, "score": score, "iterations": iterations}

def after_critique(state: State) -> str:
    if state["score"] >= PASS_SCORE or state["iterations"] > MAX_REVISIONS:
        print("[critique] Analysis approved or reached max iterations.")
        return "done"
    print("[critique] Revision requested by reviewer.")
    return router(state)

MEMORY_PATH = Path("memory/analyses.jsonl")

def save_memory(state: State) -> dict:
    """Agent Function: Learns across runs (keeps brief memories/notes)[cite: 2]."""
    record = {
        "ticker": state["ticker"],
        "saved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "route": pick_route(state["routed"]),
        "score": state["score"],
        "analysis": state["analysis"],
    }
    MEMORY_PATH.parent.mkdir(parents=True, exist_ok=True)
    with MEMORY_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"[memory] Saved {record['ticker']} run record to disk.")
    return {}

# ==========================================
# 3. BUILD GRAPH AND EXECUTE
# ==========================================
g = StateGraph(State)
g.add_node("planner", planner)
g.add_node("fetch", fetch)
g.add_node("chain", chain)
g.add_node("earnings", earnings_analyst)
g.add_node("news", news_analyst)
g.add_node("market", market_analyst)
g.add_node("critique", critique)
g.add_node("memory", save_memory)

g.set_entry_point("planner")
g.add_edge("planner", "fetch")
g.add_edge("fetch", "chain")
g.add_conditional_edges("chain", router, {"earnings": "earnings", "news": "news", "market": "market"})

for cat in ("earnings", "news", "market"):
    g.add_edge(cat, "critique")

g.add_conditional_edges("critique", after_critique, {"earnings": "earnings", "news": "news", "market": "market", "done": "memory"})
g.add_edge("memory", END)

app = g.compile()

# Execution test
out = app.invoke({"ticker": "AAPL"})

print("\n================== FINAL ANALYSIS ==================")
print(out["analysis"])
print(f"\nScore: {out['score']} after {out['iterations']} iteration(s)")

[fetch] Retrieved metrics & 9 live articles for AAPL
[router] -> Routed to earnings specialist
[critique] Round 1: Score 0.80
[critique] Analysis approved or reached max iterations.
[memory] Saved AAPL run record to disk.

================== FINAL ANALYSIS ==================
### Thesis

Apple Inc. (NASDAQ: AAPL) remains the premier cash-flow generator in the technology sector, supported by an unrivaled ecosystem of 2.5 billion active devices. Trading at $334.47 with a market capitalization of $4.88 trillion, Apple’s investment thesis has successfully transitioned from a pure hardware upgrade story to a high-margin, recurring software and services narrative. 

From an earnings perspective, the company continues to beat expectations by leverage-monetizing its massive installed base. The primary engine of earnings outperformance is the Services division, which recently surpassed 1.5 billion paid subscriptions. By shifting the revenue mix toward these high-margin subscriptions, Apple is st